<a href="https://colab.research.google.com/github/siddharth10-dev/data-viz-python/blob/main/pandas_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
%matplotlib inline

In [ ]:
%matplotlib inline

In [ ]:
pip install plotly

In [ ]:
pip install cufflinks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.6 MB/s eta 0:00:00


In [ ]:
%matplotlib inline

In [ ]:
pip install chart-studio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
from plotly import __version__
print(__version__)

5.24.1


In [ ]:
from plotly.offline import download_plotlyjs,init_notebook_mode,plot,iplot

In [ ]:
init_notebook_mode(connected=True)

In [ ]:
import cufflinks as cf
cf.go_offline()

In [ ]:
df=pd.DataFrame(np.random.randn(100,4),columns='A B C D'.split())

In [ ]:
df.head()

,A,B,C,D
0,0.590340,-1.027782,-0.563115,1.745668
1,-0.622409,0.380544,1.503156,-1.136475
2,0.263269,0.110637,-0.803539,0.847446
3,-0.005705,-0.540107,1.951732,0.221660
4,1.508391,-0.451943,1.386298,-0.818950


In [ ]:
df2=pd.DataFrame({'Category':['A','B','C'],'Values':[32,43,50]})

In [ ]:
df2

,Category,Values
0,A,32
1,B,43
2,C,50


In [ ]:
import plotly.express as px
fig = px.line(df)
fig.show()

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

fig = px.line(df, title='DataFrame Line Plot with Plotly Express')
fig.show()

### Fixing Cufflinks Compatibility
The following code patches a known bug in `cufflinks` where it fails to convert NumPy floats to Python floats when generating colors. This allows `iplot()` to work in modern environments.

In [ ]:
import cufflinks as cf
import numpy as np
import plotly.graph_objs as go
from cufflinks import colors
import re

def fix_cufflinks_final():
    # 1. Patch hex_to_rgb to handle numpy alpha values
    orig_hex_to_rgb = colors.hex_to_rgb
    def patched_hex_to_rgb(color, alpha=1):
        if hasattr(alpha, 'item'):
            alpha = alpha.item()
        return orig_hex_to_rgb(color, alpha)
    colors.hex_to_rgb = patched_hex_to_rgb

    # 2. Deep recursive cleaner for the 'np.float64' string bug
    def sanitize(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                obj[k] = sanitize(v)
            return obj
        elif isinstance(obj, list):
            return [sanitize(i) for i in obj]
        elif isinstance(obj, str) and 'np.float64' in obj:
            return re.sub(r'np\.float64\((.*?)\)', r'\1', obj)
        return obj

    # 3. Comprehensive patch for common Plotly objects used by Cufflinks
    patch_targets = [go.Scatter, go.Bar, go.Box, go.Histogram, go.Pie]

    def get_patched_init(orig_init):
        def patched_init(self, *args, **kwargs):
            new_args = [sanitize(a) for a in args]
            sanitize(kwargs)
            orig_init(self, *new_args, **kwargs)
        return patched_init

    for obj in patch_targets:
        obj.__init__ = get_patched_init(obj.__init__)

fix_cufflinks_final()
print("Global Plotly sanitization patch applied for Scatter, Bar, Box, Histogram, and Pie.")

Global Plotly sanitization patch applied for Scatter, Bar, Box, Histogram, and Pie.


Now you can use the standard `iplot()` syntax:

In [ ]:
df.iplot

<bound method _iplot of            A         B         C         D
0   0.590340 -1.027782 -0.563115  1.745668
1  -0.622409  0.380544  1.503156 -1.136475
2   0.263269  0.110637 -0.803539  0.847446
3  -0.005705 -0.540107  1.951732  0.221660
4   1.508391 -0.451943  1.386298 -0.818950
..       ...       ...       ...       ...
95 -0.164776 -0.580841  0.004224 -0.039577
96 -1.028846  1.133275 -0.273524  0.281180
97 -0.762598 -0.366027 -1.000057  1.234338
98 -0.429148  0.663532  1.454808  2.036515
99 -1.496474  2.683511 -0.216878  0.463393

[100 rows x 4 columns]>

In [ ]:
# Final test with recursive argument sanitization
df.iplot(
    kind='scatter',
    mode='lines+markers',
    size=6,
    theme='pearl',
    title='Customized Patched Cufflinks Chart',
    xTitle='Index',
    yTitle='Random Values'
)

In [ ]:
df.iplot

<bound method _iplot of            A         B         C         D
0   0.590340 -1.027782 -0.563115  1.745668
1  -0.622409  0.380544  1.503156 -1.136475
2   0.263269  0.110637 -0.803539  0.847446
3  -0.005705 -0.540107  1.951732  0.221660
4   1.508391 -0.451943  1.386298 -0.818950
..       ...       ...       ...       ...
95 -0.164776 -0.580841  0.004224 -0.039577
96 -1.028846  1.133275 -0.273524  0.281180
97 -0.762598 -0.366027 -1.000057  1.234338
98 -0.429148  0.663532  1.454808  2.036515
99 -1.496474  2.683511 -0.216878  0.463393

[100 rows x 4 columns]>

### Examples of Cufflinks Charts
Following the documentation, here are a few common chart types using the random data in `df` and the categorical data in `df2`.

In [ ]:
# 1. Bar Chart using df2
df2.iplot(kind='bar', x='Category', y='Values', title='Simple Bar Chart', theme='solar')

In [ ]:
# 2. Box Plot to show distribution of the 4 columns in df
df.iplot(kind='box', title='Box Plot of Random Data', theme='white')

In [ ]:
# 3. Histogram with custom bins
df['A'].iplot(kind='hist', bins=25, title='Histogram of Column A', xTitle='Value', yTitle='Frequency')

In [ ]:
df.iplot(kind='surface', title='3D Surface Plot', colorscale='rdylbu')

### Geographic Visualizations
For Choropleth maps, you usually need a dataset with location identifiers (like ISO country codes). Here is how you can create one using Plotly Express (recommended for maps) and Cufflinks.

In [ ]:
import plotly.express as px

# Sample data for India states (using Plotly's built-in gapminder or custom dict)
india_data = pd.DataFrame({
    'State': ['Bihar', 'Kerala', 'Punjab', 'UP'],
    'Score': [10, 20, 30, 40]
})

# Example using Plotly Express
# Note: Choropleths usually require 'locations' and 'z' (values)
fig = px.choropleth(locations=["IND"], color=[10],
                    locationmode='ISO-3',
                    title='Simple India Choropleth')
fig.show()

Alternatively, using the patched `iplot` syntax:

In [ ]:
# Note: kind='choropleth' in Cufflinks requires specific 'locations' and 'z' columns
map_df = pd.DataFrame({'code': ['USA', 'CAN', 'IND'], 'val': [10, 20, 30]})
map_df.iplot(kind='choropleth', locations='code', z='val', title='Cufflinks Choropleth Example')